In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tropycal.tracks as tracks
from paratc.tc_models import Holland1980 as h80
import os
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.io.img_tiles import OSM  # Ensure this is correctly imported
from herbie.toolbox import EasyMap, pc
from cartopy import feature as cfeature
import statistics
from datetime import datetime


In [2]:
restricted_domain = False

if restricted_domain:
    lon_min = -98
    lon_max = -81
    lat_min = 18
    lat_max = 30
else:
    # Domain range for HRRR (trapezoid shape)
    lon_min = -122.5
    lon_max = -72.5
    lat_min = 21.5
    lat_max = 47.5



gulf_bounds = {
 "min_lat": lat_min,
 "max_lat": lat_max,
 "min_lon": lon_min,
 "max_lon": lon_max
}

# Define the latitude and longitude bounds
lon_coords = [lon_min, lon_max]  # Minimum and maximum longitude for the Gulf of Mexico
lat_coords = [lat_min, lat_max]  # Minimum and maximum latitude for the Gulf of Mexico

In [3]:
# Filter storms passing through the Gulf of Mexico
def is_in_gulf(lat, lon):
 """
 Check if a single latitude/longitude point is in the Gulf of Mexico.
 """
 return (gulf_bounds['min_lat'] <= lat <= gulf_bounds['max_lat']) and \
  (gulf_bounds['min_lon'] <= lon <= gulf_bounds['max_lon'])

def is_in_range(lat, lon):
  return (lat_min <= lat <= lat_max) and \
  (lon_min <= lon <= lon_max)

def storm_passed_through_gulf(storm_df):
 """
 Check if any track point of an individual storm falls within the Gulf of Mexico.
 """
 for _, row in storm_df.iterrows():
  if is_in_gulf(row['latitude'], row['longitude']):
   return True  # At least one point in Gulf
 return False

def storm_passed_through_range(storm_df):
 """
 Check if any track point of an individual storm falls within the Gulf of Mexico.
 """
 for _, row in storm_df.iterrows():
  if is_in_range(row['latitude'], row['longitude']):
   return True  # At least one point in Gulf
 return False

In [4]:
# Get all the dataset from the Atlantic Basin
basin = tracks.TrackDataset(basin='north_atlantic')
gulf_storms = {}

for season_year in range(2014, 2024):
  season = basin.get_season(season_year)

  df = season.to_dataframe()


  # Iterate through all storms in the season
  for storm_number in df.index:
    # Get storm object
    storm = basin.get_storm((df.name[storm_number], season_year))

    # Check all track points of the storm
    passed_through_gulf = False
    for lat, lon in zip(storm.dict['lat'], storm.dict['lon']):  # Extract track points
      if is_in_range(lat, lon):  # Check if the point is in the Gulf
        passed_through_gulf = True
        break  # Stop checking once a point is within the Gulf

  # If storm passed through the Gulf, add it to the list
    if passed_through_gulf:
      gulf_storms[storm.id] = storm

# Output the storms that passed through the Gulf of Mexico
print(f"Storms that passed through the Gulf of Mexico between 2014-2023:")
for storm in gulf_storms.keys():
  print('\t' + gulf_storms[storm].id)

--> Starting to read in HURDAT2 data
--> Completed reading in HURDAT2 data (2.56 seconds)
Storms that passed through the Gulf of Mexico between 2014-2023:
	AL012014
	AL032014
	AL042014
	AL052014
	AL012015
	AL022015
	AL032015
	AL112015
	AL122015
	AL012016
	AL022016
	AL032016
	AL082016
	AL092016
	AL112016
	AL142016
	AL032017
	AL062017
	AL092017
	AL112017
	AL132017
	AL152017
	AL162017
	AL182017
	AL012018
	AL022018
	AL032018
	AL062018
	AL072018
	AL142018
	AL022019
	AL032019
	AL052019
	AL062019
	AL072019
	AL092019
	AL112019
	AL142019
	AL162019
	AL172019
	AL012020
	AL022020
	AL032020
	AL062020
	AL082020
	AL092020
	AL122020
	AL132020
	AL142020
	AL152020
	AL192020
	AL222020
	AL252020
	AL262020
	AL282020
	AL292020
	AL022021
	AL032021
	AL042021
	AL052021
	AL062021
	AL082021
	AL092021
	AL132021
	AL142021
	AL152021
	AL212021
	AL012022
	AL032022
	AL092022
	AL142022
	AL172022
	AL022023
	AL092023
	AL102023
	AL162023


In [5]:
gulf_storms[storm]

<tropycal.tracks.Storm>
Storm Summary:
    Maximum Wind:      60 knots
    Minimum Pressure:  981 hPa
    Start Time:        1800 UTC 22 September 2023
    End Time:          1800 UTC 23 September 2023

Variables:
    time        (datetime) [2023-09-21 12:00:00 .... 2023-09-24 18:00:00]
    extra_obs   (int32) [0 .... 0]
    special     (str) [ .... ]
    type        (str) [EX .... EX]
    lat         (float64) [28.5 .... 38.9]
    lon         (float64) [-76.0 .... -76.9]
    vmax        (int32) [30 .... 20]
    mslp        (int32) [1012 .... 1010]
    wmo_basin   (str) [north_atlantic .... north_atlantic]

More Information:
    id:              AL162023
    operational_id:  AL162023
    name:            OPHELIA
    year:            2023
    season:          2023
    basin:           north_atlantic
    source_info:     NHC Hurricane Database
    source:          hurdat
    ace:             1.5
    realtime:        False
    invest:          False
    subset:          False

In [6]:
from herbie import Herbie
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import os
import numpy as np

# Load herbie object for example datetime
H = Herbie(
    "2021-07-19",
    model="hrrr",
    product="prs",
    fxx=0,
    save_dir=f'./data/hrrr'
)
ds_u = H.xarray(':UGRD:10 m', backend_kwargs={'decode_timedelta': True}) # 10 meter U component of wind speed
ds_v = H.xarray(':VGRD:10 m', backend_kwargs={'decode_timedelta': True}) # 10 meter V component of wind speed
ds_p = H.xarray(':PRES:surface', backend_kwargs={'decode_timedelta': True}) # Surface Pressure


# Change coordinate system for each xarray to suit our needs
ds_u = ds_u.assign_coords(
    longitude=(((ds_u.longitude + 180) % 360) - 180)
)
ds_v = ds_v.assign_coords(
    longitude=(((ds_v.longitude + 180) % 360) - 180)
)

ds_p = ds_p.assign_coords(
    longitude=(((ds_p.longitude + 180) % 360) - 180)
)


# Subset the dataset by applying a filter on latitude and longitude using xarray's where()
subset_u = ds_u.where(
    (ds_u.latitude >= lat_min) & (ds_u.latitude <= lat_max) &
    (ds_u.longitude >= lon_min) & (ds_u.longitude <= lon_max),
    drop=True
)

subset_v = ds_v.where(
    (ds_v.latitude >= lat_min) & (ds_v.latitude <= lat_max) &
    (ds_v.longitude >= lon_min) & (ds_v.longitude <= lon_max),
    drop=True
)

subset_p = ds_p.where(
    (ds_p.latitude >= lat_min) & (ds_p.latitude <= lat_max) &
    (ds_p.longitude >= lon_min) & (ds_p.longitude <= lon_max),
    drop=True
)

# Merge the xarray data into one dataset
ds = xr.merge([subset_u, subset_v, subset_p])

✅ Found ┊ model=hrrr ┊ product=prs ┊ 2021-Jul-19 00:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws


c:\Users\joebe\anaconda3\Lib\site-packages\herbie\core.py:1112: UserWarning: Will not remove GRIB file because it previously existed.
  warnings.warn("Will not remove GRIB file because it previously existed.")


In [9]:
from datetime import datetime
from pathlib import Path
# TODO: a 66 x 600 px hrrr box then reduce using filtered_hrrr
# TODO: look into Universal Transverse Mercator (UTM)

storm_models = {}
hrrr_path = './data/filtered_hrrr'
reset = -1

hours = set([0, 6, 12, 18])

for hurr in os.listdir(hrrr_path):
    gulf_storm = gulf_storms[hurr]

    # Convert storm data to DataFrame
    track_df = pd.DataFrame({
        "time": gulf_storm.dict['time'],
        "lat": gulf_storm.dict['lat'],
        "lon": gulf_storm.dict['lon'],
        "vmax": gulf_storm.dict['vmax'],
        "pcen": gulf_storm.dict['mslp'],  # Central pressure
        "rmax": gulf_storm.dict.get('rmax', [30] * len(gulf_storm.dict['lat']))  # Default if missing
    })

    track_df = track_df[(track_df['time'].dt.hour.isin(hours)) & (track_df['time'].dt.minute == 0)]
    track_df = track_df.reset_index(drop=True)

    # Environmental pressure assumption
    track_df["penv"] = 1010.0  # Default environmental pressure
    track_df["pdelta"] = track_df["penv"] - track_df["pcen"]  # Pressure difference

    hurr_path = os.path.join(hrrr_path, hurr)
    for patch in os.listdir(hurr_path):
        patch_times = []
        patch_path = os.path.join(hurr_path, patch)
        patch_files = os.listdir(patch_path)
        loc = -1

        para_path = patch_path.replace("filtered_hrrr", "parametric_inputs")
        if os.path.isdir(para_path):
            continue

        for file in patch_files:
            timestamp_str = file.replace("_data", "")  # '05-06-2015-03:00'
            time = datetime.strptime(timestamp_str, "%m-%d-%Y-%H-%M")
            patch_times.append(time)

            time_mask = track_df['time'] == time

            if time_mask.any():
                loc = track_df.index[time_mask].tolist()[0]
            elif patch_files.index(file) == len(patch_files) - 1:
                timestamps = [dt.timestamp() for dt in patch_times]
                # Compute average timestamp
                avg_timestamp = statistics.mean(timestamps)
                # Convert back to datetime
                avg_datetime = datetime.fromtimestamp(avg_timestamp)
                loc = (track_df['time'] - avg_datetime).abs().idxmin()

            if loc != -1:
                # slice out loc-1, loc, loc+1
                # loc-1 is 6 h before, loc+1 6 h after)
                indices = [loc - 1, loc, loc + 1]
                # guard against going out of bounds
                valid = [i for i in indices if 0 <= i < len(track_df)]
                patch_df = track_df.iloc[valid]

                target_row = patch_df.loc[loc]
                target_lat, target_lon = target_row['lat'], target_row['lon']

                H = Herbie(time, model="hrrr", product="prs", fxx=0, save_dir=f'./data')
                # Fetch variables individually to avoid list issues
                patch_ds = H.xarray(':UGRD:10 m', backend_kwargs={'decode_timedelta': True}, overwrite=True)
                lon_adjusted = (patch_ds.longitude.values + 180) % 360 - 180  # Convert HRRR lon to -180
                lat = patch_ds.latitude.values


                # Calculate distance using adjusted longitude - use closest grib point
                distance_cell = np.sqrt((lat - target_lat)**2 + (lon_adjusted - target_lon)**2)
                y_idx, x_idx = np.unravel_index(np.argmin(distance_cell), distance_cell.shape)

                y_slice = slice(max(0, y_idx - 200), min(ds.y.size, y_idx + 200))
                x_slice = slice(max(0, x_idx - 200), min(ds.x.size, x_idx + 200))

                # Subset using x/y indices
                patch_ds_subset = patch_ds.isel(y=y_slice, x=x_slice)

                # Adjust longitude convention of subset
                patch_ds_subset = patch_ds_subset.assign_coords(
                    longitude=(((patch_ds_subset.longitude + 180) % 360) - 180)
                )       
                break

        # Define grid for wind field calculations
        if patch_ds_subset.longitude.ndim == 1:
            grid_lon, grid_lat = np.meshgrid(patch_ds_subset.longitude.values, patch_ds_subset.latitude.values)
        else:
            grid_lon = patch_ds_subset.longitude.values
            grid_lat = patch_ds_subset.latitude.values

        
        # Create ParaTC Holland1980 storm model
        storm_model = h80(patch_df, grid_lon, grid_lat, B_model='vickery00', interp_timestep=1)
        nt = storm_model.data['time'].size

        # Apply transformations
        storm_model.scale_winds(0.91)
        storm_model.apply_inflow_angle(inflow_model='nws')
        storm_model.add_background_winds(bg_alpha=0.55, bg_beta=20)
        storm_model.make_wind_stress(cd_model='garratt77', cd_max=3e-3)

        for file in os.listdir(patch_path):
            storm_subset = storm_model.data
            file_path = os.path.join(patch_path, file)
            with xr.open_dataset(file_path, decode_timedelta=True) as hrrr_ds:
                common_times = np.intersect1d(hrrr_ds.time.values,
                                            storm_subset.time.values)
                
                if len(common_times) == 0:
                    print(f'Missing values at: {file_path}')
                    continue
                

                storm_subset = storm_subset.sel(
                    time=hrrr_ds.time
                )

                new_path = file_path.replace("filtered_hrrr", "parametric_inputs")
                new_path = new_path.replace("\\", "/")

                # Extract and round coordinates from hrrr_ds and storm_subset
                hrrr_lats = hrrr_ds.latitude.values.ravel()
                hrrr_lons = hrrr_ds.longitude.values.ravel()
                hrrr_points = set(zip(hrrr_lats, hrrr_lons))

                storm_lats = storm_subset.lat.values.ravel()
                storm_lons = storm_subset.lon.values.ravel()

                # Create a mask indicating where (lat, lon) pairs exist in hrrr_ds
                mask = np.array([(lat, lon) in hrrr_points for lat, lon in zip(storm_lats, storm_lons)])
                # print(np.any(mask))
                mask = mask.reshape(storm_subset.lat.shape)

                # Convert the mask to an xarray DataArray with appropriate dimensions/coordinates
                mask_da = xr.DataArray(mask, dims=storm_subset.lat.dims, coords=storm_subset.lat.coords)

                # Apply the mask to crop storm_subset
                storm_subset = storm_subset.where(mask_da, drop=True)

                os.makedirs(Path(new_path).parent, exist_ok=True)
                storm_subset.to_netcdf(new_path)


✅ Found ┊ model=hrrr ┊ product=prs ┊ 2015-May-06 06:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [data\hrrr\20150506]


c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  new_time = pd.date_range( time_0[0], time_0[-1], freq=f'{new_timestep}H')
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:62: RuntimeWarning: invalid value encountered in divide
  utrans = trans_speed * utrans / vec_norm
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:63: RuntimeWarning: invalid value encountered in divide
  vtrans = trans_speed * vtrans / vec_norm


Missing values at: ./data/filtered_hrrr\AL012015\patch1\05-06-2015-03-00_data
Missing values at: ./data/filtered_hrrr\AL012015\patch1\05-06-2015-04-00_data
Missing values at: ./data/filtered_hrrr\AL012015\patch1\05-06-2015-05-00_data
✅ Found ┊ model=hrrr ┊ product=prs ┊ 2015-May-08 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [data\hrrr\20150508]


c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  new_time = pd.date_range( time_0[0], time_0[-1], freq=f'{new_timestep}H')
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:62: RuntimeWarning: invalid value encountered in divide
  utrans = trans_speed * utrans / vec_norm
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:63: RuntimeWarning: invalid value encountered in divide
  vtrans = trans_speed * vtrans / vec_norm


✅ Found ┊ model=hrrr ┊ product=prs ┊ 2015-May-08 18:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws


...

c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  new_time = pd.date_range( time_0[0], time_0[-1], freq=f'{new_timestep}H')
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:62: RuntimeWarning: invalid value encountered in divide
  utrans = trans_speed * utrans / vec_norm
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:63: RuntimeWarning: invalid value encountered in divide
  vtrans = trans_speed * vtrans / vec_norm


✅ Found ┊ model=hrrr ┊ product=prs ┊ 2020-Nov-13 00:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [data\hrrr\20201113]


c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  new_time = pd.date_range( time_0[0], time_0[-1], freq=f'{new_timestep}H')
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:62: RuntimeWarning: invalid value encountered in divide
  utrans = trans_speed * utrans / vec_norm
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:63: RuntimeWarning: invalid value encountered in divide
  vtrans = trans_speed * vtrans / vec_norm


✅ Found ┊ model=hrrr ┊ product=prs ┊ 2020-Nov-13 06:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws


c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  new_time = pd.date_range( time_0[0], time_0[-1], freq=f'{new_timestep}H')
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:62: RuntimeWarning: invalid value encountered in divide
  utrans = trans_speed * utrans / vec_norm
c:\Users\joebe\anaconda3\Lib\site-packages\paratc\track_tools.py:63: RuntimeWarning: invalid value encountered in divide
  vtrans = trans_speed * vtrans / vec_norm


In [ ]:
storm_subset

In [ ]:
# OLD
# from datetime import datetime
# from pathlib import Path
# # Do a 66 x 600 px hrrr box then reduce using filtered_hrrr

# storm_models = {}
# hrrr_path = './data/filtered_hrrr'

# for hurr in os.listdir(hrrr_path):
#     gulf_storm = gulf_storms[hurr]
    
#     # Convert storm data to DataFrame
#     track_df = pd.DataFrame({
#         "time": gulf_storm.dict['time'],
#         "lat": gulf_storm.dict['lat'],
#         "lon": gulf_storm.dict['lon'],
#         "vmax": gulf_storm.dict['vmax'],
#         "pcen": gulf_storm.dict['mslp'],  # Central pressure
#         "rmax": gulf_storm.dict.get('rmax', [30] * len(gulf_storm.dict['lat']))  # Default if missing
#     })

#     # Environmental pressure assumption
#     track_df["penv"] = 1010.0  # Default environmental pressure
#     track_df["pdelta"] = track_df["penv"] - track_df["pcen"]  # Pressure difference

#     hurr_path = os.path.join(hrrr_path, hurr)
#     for patch in os.listdir(hurr_path):
#         patch_path = os.path.join(hurr_path, patch)
#         for file in os.listdir(patch_path):
#             timestamp_str = file.replace("_data", "")  # '05-06-2015-03:00'
#             time = datetime.strptime(timestamp_str, "%m-%d-%Y-%H:%M")

#             time_mask = track_df['time'] == time
#             if time_mask.any():
#                 loc = track_df.index[time_mask].tolist()[0]

#                 # slice out loc-1, loc, loc+1
#                 # (if your data is exactly hourly, loc-1 is 6 h before, loc+1 6 h after)
#                 indices = [loc - 1, loc, loc + 1]
#                 # guard against going out of bounds
#                 valid = [i for i in indices if 0 <= i < len(track_df)]
#                 patch_df = track_df.iloc[valid]
#                 patch_ds = xr.open_dataset(os.path.join(patch_path, file), decode_timedelta=True)
#                 break

#         # Define grid for wind field calculations
#         if patch_ds.longitude.ndim == 1:
#             grid_lon, grid_lat = np.meshgrid(patch_ds.longitude.values, patch_ds.latitude.values)
#         else:
#             grid_lon = patch_ds.longitude.values
#             grid_lat = patch_ds.latitude.values

        
#         # Create ParaTC Holland1980 storm model
#         storm_model = h80(patch_df, grid_lon, grid_lat, B_model='vickery00', interp_timestep=1)
#         nt = storm_model.data['time'].size

#         # Apply transformations
#         storm_model.scale_winds(0.91)
#         storm_model.apply_inflow_angle(inflow_model='nws')
#         storm_model.add_background_winds(bg_alpha=0.55, bg_beta=20)
#         storm_model.make_wind_stress(cd_model='garratt77', cd_max=3e-3)

#         for file in os.listdir(patch_path):
#             storm_subset = storm_model.data
#             file_path = os.path.join(patch_path, file)
#             with xr.open_dataset(file_path, decode_timedelta=True) as hrrr_ds:
#                 common_times = np.intersect1d(hrrr_ds.time.values,
#                                             storm_subset.time.values)

#                 storm_subset = storm_subset.sel(
#                     time=common_times,

#                 )

#                 new_path = file_path.replace("filtered_hrrr", "parametric_inputs")

#                 os.makedirs(Path(new_path).parent, exist_ok=True)
#                 storm_subset.to_netcdf(new_path)


In [ ]:
# # OLD OLD
# for gulf_storm in gulf_storms:
#     if gulf_storm.id not in os.listdir('./data/filtered_hrrr'):
#         continue
#     print(gulf_storm.id)
#     # Convert storm data to DataFrame
#     track_df = pd.DataFrame({
#         "time": gulf_storm.dict['time'],
#         "lat": gulf_storm.dict['lat'],
#         "lon": gulf_storm.dict['lon'],
#         "vmax": gulf_storm.dict['vmax'],
#         "pcen": gulf_storm.dict['mslp'],  # Central pressure
#         "rmax": gulf_storm.dict.get('rmax', [30] * len(gulf_storm.dict['lat']))  # Default if missing
#     })
#     # Environmental pressure assumption
#     track_df["penv"] = 1010.0  # Default environmental pressure
#     track_df["pdelta"] = track_df["penv"] - track_df["pcen"]  # Pressure difference

#     storm_lon_min = gulf_storm.lon.min() - 3
#     storm_lon_max = gulf_storm.lon.max() + 3
#     storm_lat_min = gulf_storm.lat.min() - 3
#     storm_lat_max = gulf_storm.lat.max() + 3

#     # Combine masks for both bounding boxes
#     combined_mask = (
#         (ds.longitude >= max(lon_min, storm_lon_min)) & 
#         (ds.longitude <= min(lon_max, storm_lon_max)) &
#         (ds.latitude >= max(lat_min, storm_lat_min)) & 
#         (ds.latitude <= min(lat_max, storm_lat_max))
#     )

#     ds_subset = ds.where(combined_mask, drop=True)

#     # Define grid for wind field calculations
#     if ds_subset.longitude.ndim == 1:
#         grid_lon, grid_lat = np.meshgrid(ds_subset.longitude.values, ds_subset.latitude.values)
#     else:
#         grid_lon = ds_subset.longitude.values
#         grid_lat = ds_subset.latitude.values


#     # Create ParaTC Holland1980 storm model
#     storm_model = h80(track_df, grid_lon, grid_lat, B_model='vickery00', interp_timestep=1)
#     nt = storm_model.data['time'].size

#     # Apply transformations
#     storm_model.scale_winds(0.91)
#     storm_model.apply_inflow_angle(inflow_model='nws')
#     storm_model.add_background_winds(bg_alpha=0.55, bg_beta=20)
#     storm_model.make_wind_stress(cd_model='garratt77', cd_max=3e-3)

#     storm_models[gulf_storm.id] = storm_model

In [ ]:
storm_models.keys()

In [ ]:
storm_models['AL012015'].data

In [ ]:
import os
import xarray as xr
from pathlib import Path

path = './data/filtered_hrrr/AL012015'
for root, _, points in os.walk(path):
    for point in points:
        relative_path = os.path.join(root, point)
        storm_id = Path(relative_path).parts[2]
        new_path = relative_path.replace("filtered_HRRR", "parametric_inputs")
        print(new_path)
        with xr.open_dataset(relative_path, decode_timedelta=True) as hrrr_ds:
            lat0, lat1 = hrrr_ds.latitude.min().item(), hrrr_ds.latitude.max().item()
            lon0, lon1 = hrrr_ds.longitude.min().item(), hrrr_ds.longitude.max().item()

            lat2d = storm_model.data["lat"]
            lon2d = storm_model.data["lon"]

            mask = ((lat2d >= lat0) & (lat2d <= lat1) &
                    (lon2d >= lon0) & (lon2d <= lon1))

            #storm_subset = storm_models[storm_id].data.where(mask, drop=True)
            storm_subset = storm_models[storm_id].data

            common_lats = np.intersect1d(hrrr_ds.latitude.values,
                                        storm_subset.lat.values)

            common_lons =  np.intersect1d(hrrr_ds.longitude.values,
                                        storm_subset.lon.values)

            common_times = np.intersect1d(hrrr_ds.time.values,
                                        storm_subset.time.values)

            storm_subset = storm_subset.sel(
                time=common_times,
                lat=common_lats,

            )

            os.makedirs(Path(new_path).parent, exist_ok=True)
            storm_subset.to_netcdf(new_path)

In [ ]:
import os
import xarray as xr
from pathlib import Path
cwd = os.getcwd()
data_path = './data/filtered_HRRR'
for root, _, points in os.walk(data_path):
    for point in points:
        relative_path = os.path.join(root, point)
        storm_id = Path(relative_path).parts[2]
        new_path = relative_path.replace("filtered_HRRR", "parametric_inputs")
        print(new_path)
        with xr.open_dataset(relative_path, decode_timedelta=True) as hrrr_ds:
            lat0, lat1 = hrrr_ds.latitude.min().item(), hrrr_ds.latitude.max().item()
            lon0, lon1 = hrrr_ds.longitude.min().item(), hrrr_ds.longitude.max().item()

            lat2d = storm_model.data["lat"]
            lon2d = storm_model.data["lon"]

            mask = ((lat2d >= lat0) & (lat2d <= lat1) &
                    (lon2d >= lon0) & (lon2d <= lon1))

            storm_subset = storm_models[storm_id].data.where(mask, drop=True)

            common_times = np.intersect1d(hrrr_ds.time.values,
                                        storm_subset.time.values)

            storm_subset = storm_subset.sel(
                time=common_times,
            )

            os.makedirs(Path(new_path).parent, exist_ok=True)
            storm_subset.to_netcdf(new_path)

In [ ]:
storm_subset

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.PlateCarree()})

windspeed = storm_subset['windspeed'][0].values
pressure = storm_subset['pressure'][0].values
lon = storm_subset['lon'].values
lat = storm_subset['lat'].values

im = ax.contourf(lon, lat, windspeed, transform=ccrs.PlateCarree(), cmap='coolwarm', alpha=0.6)
plt.colorbar(im, ax=ax, orientation='vertical', label='Wind Speed (m/s)')
time_value = storm_subset['time'][0].values
if isinstance(time_value, np.datetime64):
    time_value = pd.Timestamp(time_value)  # Pandas Timestamp works with strftime
ax.set_title(f"Wind Speed - Time: {time_value.strftime('%Y-%m-%d %H:%M')}")
ax.set_extent([-93, -77, 21, 35], ccrs.PlateCarree())
osm_tiles = OSM()
ax.add_image(osm_tiles, 8)
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels = gl.right_labels = False
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER

plt.show()